In [76]:
using DataFrames
using CSV
using DelimitedFiles

var_df = CSV.read("blue_vars.csv", DataFrame);

In [3]:
archive_df = CSV.read("spear_catalog_blue.csv", DataFrame)
decp_df = CSV.read("decp_blue.csv", DataFrame)
;

In [ ]:
function search(df, column, value)
    df[df[!,column] .== value, :]
end

function search(df, col_val_dict::Dict)
    truths = ones(Bool, size(df)[1])
    for (k,v) in col_val_dict
        @. truths &= (df[!,k] == v)
    end
    return df[truths,:]
end

function search(df, col_val_df::DataFrameRow)
    truths = ones(Bool, size(df)[1])
    for n in names(col_val_df)
        @. truths &= (df[!,n] == col_val_df[n])
    end
    return df[truths,:]
end

function get_files(adf, ddf, vdf)
    afiles = Vector{String}()
    dfiles = Vector{String}()

    for x in eachrow(vdf)
        arch_search = search(adf, x)
        times_a = unique(arch_search[!,:time_range])

        decp_search = search(ddf, x)
        times_d = unique(decp_search[!,:time_range])

        for ta in times_a
            if ta in times_d
                append!(dfiles, decp_search[decp_search[!,:time_range] .== ta, :path])
            else
                append!(afiles, arch_search[arch_search[!,:time_range] .== ta, :path])
            end
        end
    end
    return (afiles, dfiles)
end

get_files (generic function with 1 method)

In [68]:
archive_files, decp_files = get_files(archive_df, decp_df, var_df)

(["/archive/wfc/SPEAR/SPEAR_c192_o1_Hist_AllForc_IC1921_K50_ens_01_03/pp_ens_01/ocean/ts/monthly/10yr/ocean.192101-193012.ssh.nc", "/archive/wfc/SPEAR/SPEAR_c192_o1_Hist_AllForc_IC1921_K50_ens_01_03/pp_ens_02/ocean/ts/monthly/10yr/ocean.192101-193012.ssh.nc", "/archive/wfc/SPEAR/SPEAR_c192_o1_Hist_AllForc_IC1921_K50_ens_01_03/pp_ens_03/ocean/ts/monthly/10yr/ocean.192101-193012.ssh.nc", "/archive/wfc/SPEAR/SPEAR_c192_o1_Hist_AllForc_IC1921_K50_ens_04_06/pp_ens_01/ocean/ts/monthly/10yr/ocean.192101-193012.ssh.nc", "/archive/wfc/SPEAR/SPEAR_c192_o1_Hist_AllForc_IC1921_K50_ens_04_06/pp_ens_02/ocean/ts/monthly/10yr/ocean.192101-193012.ssh.nc", "/archive/wfc/SPEAR/SPEAR_c192_o1_Hist_AllForc_IC1921_K50_ens_04_06/pp_ens_03/ocean/ts/monthly/10yr/ocean.192101-193012.ssh.nc", "/archive/wfc/SPEAR/SPEAR_c192_o1_Hist_AllForc_IC1921_K50_ens_07_09/pp_ens_01/ocean/ts/monthly/10yr/ocean.192101-193012.ssh.nc", "/archive/wfc/SPEAR/SPEAR_c192_o1_Hist_AllForc_IC1921_K50_ens_07_09/pp_ens_02/ocean/ts/monthly/

In [77]:
writedlm("archive_files.txt", archive_files)
writedlm("decp_files.txt", decp_files)